In [ ]:
import os
import shutil
import numpy as np
from config import Config
from env import MySumoEnv

_NEEDED_WORKER_FILES = ("detectors.add.xml", "network.net.xml", "run.sumocfg")

def prepare_worker_dir(worker_idx=0):
    if hasattr(Config, "ensure_dirs"):
        Config.ensure_dirs()
    else:
        os.makedirs(Config.WORKERS_ROOT, exist_ok=True)
    src = Config.BASE_WORK_DIR
    dst = Config.worker_dir(worker_idx)
    os.makedirs(dst, exist_ok=True)
    for name in _NEEDED_WORKER_FILES:
        src_path = os.path.join(src, name)
        dst_path = os.path.join(dst, name)
        if not os.path.exists(src_path):
            raise FileNotFoundError(f"No utils file: {src_path}")
        shutil.copy2(src_path, dst_path)
    os.makedirs(os.path.join(dst, "dump"), exist_ok=True)
    return dst

def build_env(worker_idx=0, total_step=None, seed=42):
    rl_dir = prepare_worker_dir(worker_idx)
    answer_path = Config.answer_path_for(worker_idx)
    sumocfg_path = os.path.join(rl_dir, "run.sumocfg")

    if not os.path.exists(rl_dir):
        raise FileNotFoundError(f"No directory: {rl_dir}")
    if not os.path.exists(sumocfg_path):
        raise FileNotFoundError(f"No run.sumocfg: {sumocfg_path}")
    if not os.path.exists(answer_path):
        raise FileNotFoundError(f"No answer.csv: {answer_path}")

    if total_step is None:
        total_step = Config.TOTAL_STEP

    env = MySumoEnv(
        rl_dir=rl_dir,
        sumo_binary=Config.SUMO_BINARY,
        origin_list=Config.ORIGIN_LIST,
        destination_list=Config.DESTINATION_LIST,
        input_interval=Config.INPUT_INTERVAL,
        detector_interval=Config.DETECTOR_INTERVAL,
        num_OD=Config.NUM_OD,
        state_dim=1,
        answer_dir=answer_path,
        total_step=int(total_step),
    )

    obs, info = env.reset(seed=seed)
    return env, obs, info

In [ ]:
import os
import sys
import json
import time
import tempfile
import subprocess
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
from config import Config
from trial_timing import reset_trial_times, append_trial_time

TRIALS = list(range(5))
MAX_WORKERS = len(TRIALS)
_trial_result_dir = RESULT_DIR if "RESULT_DIR" in globals() else Config.RESULT_DIR
os.makedirs(Config.RESULT_DIR, exist_ok=True)
reset_trial_times(_trial_result_dir)

_runner_source = "import os\nimport sys\nimport json\nimport time\nfrom pathlib import Path\n_scripts_dir = Path(os.environ['DODE_SCRIPTS_DIR']).resolve()\nos.chdir(_scripts_dir)\nif str(_scripts_dir) not in sys.path:\n    sys.path.insert(0, str(_scripts_dir))\ntrial = int(sys.argv[1])\n\nimport os\nimport shutil\nimport numpy as np\nfrom config import Config\nfrom env import MySumoEnv\n\n_NEEDED_WORKER_FILES = (\"detectors.add.xml\", \"network.net.xml\", \"run.sumocfg\")\n\ndef prepare_worker_dir(worker_idx=0):\n    if hasattr(Config, \"ensure_dirs\"):\n        Config.ensure_dirs()\n    else:\n        os.makedirs(Config.WORKERS_ROOT, exist_ok=True)\n    src = Config.BASE_WORK_DIR\n    dst = Config.worker_dir(worker_idx)\n    os.makedirs(dst, exist_ok=True)\n    for name in _NEEDED_WORKER_FILES:\n        src_path = os.path.join(src, name)\n        dst_path = os.path.join(dst, name)\n        if not os.path.exists(src_path):\n            raise FileNotFoundError(f\"No utils file: {src_path}\")\n        shutil.copy2(src_path, dst_path)\n    os.makedirs(os.path.join(dst, \"dump\"), exist_ok=True)\n    return dst\n\ndef build_env(worker_idx=0, total_step=None, seed=42):\n    rl_dir = prepare_worker_dir(worker_idx)\n    answer_path = Config.answer_path_for(worker_idx)\n    sumocfg_path = os.path.join(rl_dir, \"run.sumocfg\")\n\n    if not os.path.exists(rl_dir):\n        raise FileNotFoundError(f\"No directory: {rl_dir}\")\n    if not os.path.exists(sumocfg_path):\n        raise FileNotFoundError(f\"No run.sumocfg: {sumocfg_path}\")\n    if not os.path.exists(answer_path):\n        raise FileNotFoundError(f\"No answer.csv: {answer_path}\")\n\n    if total_step is None:\n        total_step = Config.TOTAL_STEP\n\n    env = MySumoEnv(\n        rl_dir=rl_dir,\n        sumo_binary=Config.SUMO_BINARY,\n        origin_list=Config.ORIGIN_LIST,\n        destination_list=Config.DESTINATION_LIST,\n        input_interval=Config.INPUT_INTERVAL,\n        detector_interval=Config.DETECTOR_INTERVAL,\n        num_OD=Config.NUM_OD,\n        state_dim=1,\n        answer_dir=answer_path,\n        total_step=int(total_step),\n    )\n\n    obs, info = env.reset(seed=seed)\n    return env, obs, info\n\nimport time\nfrom trial_timing import reset_trial_times, append_trial_time\n\nimport os\nimport json\nimport numpy as np\nfrom config import Config\n\ndef evenly_binary_sequence(count: int, n: int) -> np.ndarray:\n    \"\"\"\n    Create a 0/1 sequence of length n with count ones distributed as evenly as possible.\n    \"\"\"\n    count = int(np.clip(count, 0, n))\n    seq = np.zeros(n, dtype=np.int32)\n    if count == 0:\n        return seq\n    if count == n:\n        seq[:] = 1\n        return seq\n\n    acc = 0\n    for i in range(n):\n        acc += count\n        if acc >= n:\n            seq[i] = 1\n            acc -= n\n    return seq\n\ndef counts_to_binary_action_plan(\n    counts_block: np.ndarray,\n    num_od: int,\n    total_step: int,\n    block_steps: int\n) -> np.ndarray:\n    \"\"\"\n    counts_block: shape [n_blocks, num_od], where each value is the total count for the corresponding 300-second block (0-60).\n    Returns: shape [total_step, num_od], where each 5-second step action is 0/1.\n    \"\"\"\n    n_blocks = int(np.ceil(total_step / block_steps))\n    action_plan = np.zeros((total_step, num_od), dtype=np.int32)\n\n    for b in range(n_blocks):\n        s = b * block_steps\n        e = min((b + 1) * block_steps, total_step)\n        n_local = e - s\n        for od in range(num_od):\n            c = int(np.rint(counts_block[b, od]))\n            c = int(np.clip(c, 0, n_local))\n            action_plan[s:e, od] = evenly_binary_sequence(c, n_local)\n\n    return action_plan\n\ndef run_episode_with_plan(build_env_fn, worker_idx, action_plan, seed):\n    env, obs, info = build_env_fn(\n        worker_idx=worker_idx,\n        total_step=action_plan.shape[0],\n        seed=seed\n    )\n    total_reward = 0.0\n    last_info = {}\n\n    try:\n        for t in range(action_plan.shape[0]):\n            obs, reward, terminated, truncated, info = env.step(action_plan[t])\n            total_reward += float(reward)\n            last_info = info\n            if terminated or truncated:\n                break\n    finally:\n        env.close()\n\n    trajectory = last_info.get(\"trajectory\", [])\n    return float(total_reward), trajectory\n\ndef cmaes_optimize_counts_300s(\n    build_env_fn,\n    worker_idx,\n    num_od,\n    total_step,\n    input_interval,\n    detector_interval,\n    init_points=0,\n    n_iter=200,\n    seed=42,\n    verbose=2,\n    popsize=16,\n    sigma0=12.0,\n):\n    block_steps = detector_interval // input_interval\n    n_blocks = int(np.ceil(total_step / block_steps))\n\n    var_meta = []\n    lb = []\n    ub = []\n    for b in range(n_blocks):\n        s = b * block_steps\n        e = min((b + 1) * block_steps, total_step)\n        n_local = e - s\n        for od in range(num_od):\n            var_meta.append((b, od, n_local))\n            lb.append(0.0)\n            ub.append(float(n_local))\n\n    lb = np.array(lb, dtype=np.float64)\n    ub = np.array(ub, dtype=np.float64)\n    n = lb.size\n\n    rng = np.random.default_rng(seed)\n    eval_budget = int(max(1, init_points + n_iter))\n    eval_count = 0\n\n    def x_to_counts(x_vec: np.ndarray) -> np.ndarray:\n        x_vec = np.asarray(x_vec, dtype=np.float64)\n        counts = np.zeros((n_blocks, num_od), dtype=np.float64)\n        for i, (b, od, n_local) in enumerate(var_meta):\n            counts[b, od] = float(np.clip(x_vec[i], 0.0, float(n_local)))\n        return counts\n\n    def evaluate_x(x_vec: np.ndarray):\n        nonlocal eval_count\n        x_vec = np.clip(np.asarray(x_vec, dtype=np.float64), lb, ub)\n        counts_block = x_to_counts(x_vec)\n        action_plan = counts_to_binary_action_plan(\n            counts_block=counts_block,\n            num_od=num_od,\n            total_step=total_step,\n            block_steps=block_steps\n        )\n        score, _ = run_episode_with_plan(\n            build_env_fn=build_env_fn,\n            worker_idx=worker_idx,\n            action_plan=action_plan,\n            seed=seed + eval_count,\n        )\n        eval_count += 1\n        return float(score)\n\n    lam = int(max(4, popsize))\n    mu = lam // 2\n\n    w = np.log(mu + 0.5) - np.log(np.arange(1, mu + 1))\n    w = w / np.sum(w)\n    mueff = 1.0 / np.sum(w ** 2)\n\n    cc = (4 + mueff / n) / (n + 4 + 2 * mueff / n)\n    cs = (mueff + 2) / (n + mueff + 5)\n    c1 = 2.0 / ((n + 1.3) ** 2 + mueff)\n    cmu = min(1.0 - c1, 2.0 * (mueff - 2.0 + 1.0 / mueff) / ((n + 2.0) ** 2 + mueff))\n    damps = 1.0 + 2.0 * max(0.0, np.sqrt((mueff - 1.0) / (n + 1.0)) - 1.0) + cs\n    chi_n = np.sqrt(n) * (1.0 - 1.0 / (4.0 * n) + 1.0 / (21.0 * n * n))\n\n    m = 0.5 * (lb + ub)\n    sigma = float(sigma0)\n    C = np.eye(n, dtype=np.float64)\n    pc = np.zeros(n, dtype=np.float64)\n    ps = np.zeros(n, dtype=np.float64)\n\n    B = np.eye(n, dtype=np.float64)\n    D = np.ones(n, dtype=np.float64)\n    inv_sqrt_C = np.eye(n, dtype=np.float64)\n\n    best_score = -np.inf\n    best_x = m.copy()\n    history = []\n    gen = 0\n\n    def _update_eigendecomp(C_mat):\n\n        C_sym = 0.5 * (C_mat + C_mat.T)\n        eigvals, eigvecs = np.linalg.eigh(C_sym)\n        eigvals = np.clip(eigvals, 1e-20, None)\n        D_new = np.sqrt(eigvals)\n        B_new = eigvecs\n        inv_sqrt_C_new = B_new @ np.diag(1.0 / D_new) @ B_new.T\n        return C_sym, B_new, D_new, inv_sqrt_C_new\n\n    C, B, D, inv_sqrt_C = _update_eigendecomp(C)\n\n    while eval_count + lam <= eval_budget:\n        gen += 1\n\n        arz = rng.standard_normal((lam, n))\n        BD = B * D\n        ary = arz @ BD.T\n        arx = m[None, :] + sigma * ary\n        arx = np.clip(arx, lb[None, :], ub[None, :])\n\n        fitness = np.empty(lam, dtype=np.float64)\n        for i in range(lam):\n            fitness[i] = evaluate_x(arx[i])\n            if fitness[i] > best_score:\n                best_score = float(fitness[i])\n                best_x = arx[i].copy()\n\n        idx = np.argsort(-fitness)\n        x_sel = arx[idx[:mu]]\n        y_sel = ary[idx[:mu]]\n        z_sel = arz[idx[:mu]]\n\n        y_w = np.sum(w[:, None] * y_sel, axis=0)\n        z_w = np.sum(w[:, None] * z_sel, axis=0)\n        m = m + sigma * y_w\n        m = np.clip(m, lb, ub)\n\n        ps = (1 - cs) * ps + np.sqrt(cs * (2 - cs) * mueff) * (inv_sqrt_C @ y_w)\n        norm_ps = np.linalg.norm(ps)\n\n        hsig_lhs = norm_ps / np.sqrt(1 - (1 - cs) ** (2 * gen)) / chi_n\n        hsig = 1.0 if hsig_lhs < (1.4 + 2.0 / (n + 1.0)) else 0.0\n\n        pc = (1 - cc) * pc + hsig * np.sqrt(cc * (2 - cc) * mueff) * y_w\n\n        rank_one = np.outer(pc, pc)\n        rank_mu = (y_sel * w[:, None]).T @ y_sel\n        C = (1 - c1 - cmu) * C + c1 * (rank_one + (1 - hsig) * cc * (2 - cc) * C) + cmu * rank_mu\n\n        sigma = sigma * np.exp((cs / damps) * (norm_ps / chi_n - 1.0))\n        sigma = float(np.clip(sigma, 1e-6, 50.0))\n\n        if gen == 1 or gen % max(1, n // 10) == 0:\n            C, B, D, inv_sqrt_C = _update_eigendecomp(C)\n\n        history.append({\n            \"gen\": int(gen),\n            \"eval\": int(eval_count),\n            \"best_score\": float(best_score),\n            \"mean_fitness\": float(np.mean(fitness)),\n            \"sigma\": float(sigma),\n        })\n\n        if verbose >= 2 and (gen == 1 or gen % 5 == 0):\n            print(f\"[CMA-ES] gen {gen} | eval {eval_count}/{eval_budget} | best={best_score:.6f} | sigma={sigma:.4f}\")\n\n    while eval_count < eval_budget:\n        xr = best_x + rng.normal(0.0, 1.0, size=n) * (0.2 * sigma + 1.0)\n        xr = np.clip(xr, lb, ub)\n        yr = evaluate_x(xr)\n        history.append({\"eval\": int(eval_count), \"phase\": \"tail_local\", \"score\": float(yr)})\n        if yr > best_score:\n            best_score = float(yr)\n            best_x = xr.copy()\n\n    best_counts = x_to_counts(best_x)\n    best_actions = counts_to_binary_action_plan(\n        counts_block=best_counts,\n        num_od=num_od,\n        total_step=total_step,\n        block_steps=block_steps\n    )\n\n    cma_log = {\n        \"method\": \"CMA-ES\",\n        \"dim\": int(n),\n        \"eval_budget\": int(eval_budget),\n        \"total_evals\": int(eval_count),\n        \"popsize\": int(lam),\n        \"mu\": int(mu),\n        \"sigma0\": float(sigma0),\n        \"sigma_final\": float(sigma),\n        \"generations\": int(gen),\n        \"cc\": float(cc),\n        \"cs\": float(cs),\n        \"c1\": float(c1),\n        \"cmu\": float(cmu),\n        \"damps\": float(damps),\n        \"history\": history,\n    }\n\n    return best_actions, float(best_score), cma_log, best_counts\n\nos.makedirs(Config.RESULT_DIR, exist_ok=True)\n\n_trial_result_dir = RESULT_DIR if \"RESULT_DIR\" in globals() else Config.RESULT_DIR\n\n_trial_t0 = time.perf_counter()\ntrial_idx = trial\nseed_opt = int(100 + (trial_idx + 1))\n# Estimate seed (101-105) is kept separate from reproduce seeds (0-4).\nseed_eval = trial\n\nbest_actions, best_score, cma_log, best_counts = cmaes_optimize_counts_300s(\n    build_env_fn=build_env,\n    worker_idx=trial,\n    num_od=Config.NUM_OD,\n    total_step=Config.TOTAL_STEP,\n    input_interval=Config.INPUT_INTERVAL,\n    detector_interval=Config.DETECTOR_INTERVAL,\n    init_points=0,\n    n_iter=600,\n    seed=seed_opt,\n    verbose=2,\n    popsize=16,\n    sigma0=12.0,\n)\n\nprint(f\"[trial {trial}] Best total_reward:\", best_score)\nprint(f\"[trial {trial}] Best action shape:\", best_actions.shape)\nprint(f\"[trial {trial}] Best counts shape:\", best_counts.shape)\n\nreplay_reward, trajectory = run_episode_with_plan(\n    build_env_fn=build_env,\n    worker_idx=trial,\n    action_plan=best_actions,\n    seed=seed_eval\n)\nprint(f\"[trial {trial}] Replay reward:\", replay_reward)\n\nwith open(os.path.join(Config.RESULT_DIR, f\"trajectory_{trial}.json\"), \"w\", encoding=\"utf-8\") as f:\n    json.dump(\n        trajectory, f, ensure_ascii=False, indent=2,\n        default=lambda o: o.tolist() if isinstance(o, np.ndarray) else o\n    )\n\nwith open(os.path.join(Config.RESULT_DIR, f\"cma_log_{trial}.json\"), \"w\", encoding=\"utf-8\") as f:\n    json.dump(cma_log, f, ensure_ascii=False, indent=2)\n\n_elapsed_path = os.environ.get('DODE_TRIAL_ELAPSED_PATH')\nif _elapsed_path:\n    with open(_elapsed_path, 'w', encoding='utf-8') as f:\n        json.dump({'trial': int(trial), 'elapsed_sec': float(time.perf_counter() - _trial_t0)}, f)\n"
_scripts_dir = Path(Config.RL_ROOT).resolve() / "scripts"

def _run_trial_process(trial, runner_path, tmp_dir):
    elapsed_path = Path(tmp_dir) / f"elapsed_{trial}.json"
    env = os.environ.copy()
    env["DODE_SCRIPTS_DIR"] = str(_scripts_dir)
    env["DODE_TRIAL_ELAPSED_PATH"] = str(elapsed_path)
    env["PYTHONUNBUFFERED"] = "1"
    cmd = [sys.executable, str(runner_path), str(trial)]
    started = time.perf_counter()
    print(f"[trial {trial}] started")
    proc = subprocess.run(cmd, cwd=str(_scripts_dir), env=env)
    elapsed = time.perf_counter() - started
    if proc.returncode != 0:
        raise RuntimeError(f"trial {trial} failed with exit code {proc.returncode}")
    if elapsed_path.exists():
        with open(elapsed_path, "r", encoding="utf-8") as f:
            payload = json.load(f)
        elapsed = float(payload.get("elapsed_sec", elapsed))
    return int(trial), float(elapsed)

with tempfile.TemporaryDirectory(prefix="dode_main_parallel_") as _tmp_dir:
    _runner_path = Path(_tmp_dir) / "trial_runner.py"
    _runner_path.write_text(_runner_source, encoding="utf-8")

    _results = []
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = {executor.submit(_run_trial_process, trial, _runner_path, _tmp_dir): trial for trial in TRIALS}
        for future in as_completed(futures):
            trial, elapsed = future.result()
            print(f"[trial {trial}] completed in {elapsed:.3f} sec")
            _results.append((trial, elapsed))

for trial, elapsed in sorted(_results):
    append_trial_time(_trial_result_dir, trial, elapsed)

In [ ]:
import os
import shutil
from config import Config


def remove_workers_root():
                                                            
    root = os.path.abspath(Config.WORKERS_ROOT)
    project_root = os.path.abspath(Config.RL_ROOT)
    if os.path.basename(root) != "workers":
        raise RuntimeError(f"Refusing to delete non-workers path: {root}")
    if os.path.commonpath([root, project_root]) != project_root:
        raise RuntimeError(f"Refusing to delete path outside experiment root: {root}")
    if os.path.isdir(root):
        shutil.rmtree(root)


remove_workers_root()
